In [1]:
pip install pandas numpy google-cloud-bigquery ipykernel db-dtypes


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd
import numpy as np

from google.cloud import bigquery

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/google/api_core/_python_version_support.py:263: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [3]:
PROJECT_ID = "tourism-pipeline" 
MART_DATASET = "mart"
MART_TABLE = "visitor_prediction_features"
BQ_LOCATION = "asia-southeast1"

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../keys/gcp-key.json"

client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
print("Connected to BigQuery")

query = f"""
SELECT *
FROM `{PROJECT_ID}.{MART_DATASET}.{MART_TABLE}`
ORDER BY country, month
"""

df = client.query(query).to_dataframe()
print(df.shape)
df.head()

Connected to BigQuery


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


(9104, 10)


,country,month,visitor_arrivals,gdp,exchange_rate,public_holiday_count,aircraft_passengers,traffic_volume,hotel_rate,hotel_occupancy
0,Australia,1978-01-01,20379.0,135608.0,NaN,2,NaN,NaN,NaN,NaN
1,Australia,1978-02-01,18852.0,135608.0,NaN,0,NaN,NaN,NaN,NaN
2,Australia,1978-03-01,20819.0,135608.0,NaN,9,NaN,NaN,NaN,NaN
3,Australia,1978-04-01,18697.0,135608.0,NaN,1,NaN,NaN,NaN,NaN
4,Australia,1978-05-01,19797.0,135608.0,NaN,3,NaN,NaN,NaN,NaN


In [4]:
# Make a copy
data = df.copy()

# Parse month
data["month"] = pd.to_datetime(data["month"])

# Convert numeric columns where possible
for col in data.columns:
    if col not in ["country", "month"]:
        data[col] = pd.to_numeric(data[col], errors="coerce")

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9104 entries, 0 to 9103
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   country               9104 non-null   object        
 1   month                 9104 non-null   datetime64[ns]
 2   visitor_arrivals      9104 non-null   float64       
 3   gdp                   9070 non-null   float64       
 4   exchange_rate         3596 non-null   float64       
 5   public_holiday_count  9104 non-null   Int64         
 6   aircraft_passengers   8720 non-null   float64       
 7   traffic_volume        3840 non-null   float64       
 8   hotel_rate            3344 non-null   float64       
 9   hotel_occupancy       3344 non-null   float64       
dtypes: Int64(1), datetime64[ns](1), float64(7), object(1)
memory usage: 720.3+ KB


In [5]:
print("Rows, Columns:", data.shape)
print("\nColumns:")
print(list(data.columns))

print("\nDate range:")
print(data["month"].min(), "to", data["month"].max())

print("\nNumber of countries:")
print(data["country"].nunique())

Rows, Columns: (9104, 10)

Columns:
['country', 'month', 'visitor_arrivals', 'gdp', 'exchange_rate', 'public_holiday_count', 'aircraft_passengers', 'traffic_volume', 'hotel_rate', 'hotel_occupancy']

Date range:
1978-01-01 00:00:00 to 2025-05-01 00:00:00

Number of countries:
16


In [7]:
output_path = "../data/visitor_prediction_features_snapshot.csv"
data.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

Saved to ../data/visitor_prediction_features_snapshot.csv
